In [1]:
import torch
import pickle
import numpy as np

In [22]:
PTH = r'.\picklefiles'
CSV = r'.\csvfiles'
with open(f"{PTH}\\eal_test.pkl", 'rb') as eal:
    data = pickle.load(eal)

ent = [data[i][0] for i in range(len(data))]

asp = [data[i][1] for i in range(len(data))]

In [3]:
PTH = r".\picklefiles"
CSV = r'.\csvfiles'
def read_tensor(filename):
    with open(f'{PTH}\\{filename}', 'rb') as f:
        emb = np.load(f, allow_pickle = True)
    return emb

In [4]:
ent_emb = read_tensor('targetentemb_test.pkl').cpu()
asp_emb = read_tensor('aspentemb_test.pkl').cpu()
context_emb = read_tensor('contextemb_test.pkl').cpu()

In [5]:
aspect = [el[1] for el in data]
lngth = [len(el['candidate_aspects']) for el in aspect]
asp_count = sum([len(el['candidate_aspects']) for el in aspect])

In [8]:
#Case where true aspect is not in the candidate aspect as it is

i = 0
j = 0
check = 0
yo = 0
unid_ls = []
while(i < asp_count):
    aspect = asp[j]['true_aspect']
    i+=1    
    this = 0
    for cand in asp[j]['candidate_aspects']:
        if i >= asp_count:
            break
        if cand['aspect_name'] == aspect:
            check += 1
        if cand['aspect_name'] != aspect:
            this += 1
            casp = cand['aspect_name']
            i+=1
    if this == len(asp[j]['candidate_aspects']):
        yo += 1  
        unid_ls.append(j)  
            
    j += 1

In [10]:
#Case where true aspect is not in the candidate aspect as it is and i is stored in the list
i = 0
j = 0
ind = []
while(i < asp_count):
    aspect = asp[j]['true_aspect']
    i = i+1
    this = 0
    for cand in asp[j]["candidate_aspects"]:
        if i >= asp_count:
            break
        if j in unid_ls and aspect in str(cand["aspect_name"]):
            ind.append(i)
            #i = i+1
            continue
        if aspect != cand['aspect_name']:
            i = i+1
    j = j+1

In [12]:
mp = {}
for i in range(len(unid_ls)):
    mp[ind[i]] = unid_ls[i]

In [15]:
#Building the dataset for entity and asepct and context
test = []
dataset = np.zeros((asp_count, 201 + context_emb.shape[1]))
i = 0
k = 0
for _ in range(len(dataset)):
    while(k < len(lngth)):
        aspname = asp[k]['true_aspect']
        dataset[i, : ent_emb.shape[1]] = ent_emb[k]
        dataset[i, ent_emb.shape[1] : ent_emb.shape[1] + context_emb.shape[1]] = context_emb[k]
        dataset[i, ent_emb.shape[1] + context_emb.shape[1] : -1] = asp_emb[i]
        dataset[i, -1] = 1
        test.append((ent[k]['o_id'], asp[k]['true_aspect_id'], 1))
        #print(i, k, 'first')
        i += 1
        for p in range(lngth[k]):
            if k in unid_ls and aspname in asp[k]['candidate_aspects'][p]['aspect_name']:
                #print(k)
                continue
            if asp[k]['true_aspect'] != asp[k]['candidate_aspects'][p]['aspect_name']:
                dataset[i, : ent_emb.shape[1]] = ent_emb[k]
                dataset[i, ent_emb.shape[1] : ent_emb.shape[1] + context_emb.shape[1]] = context_emb[k]
                dataset[i, ent_emb.shape[1] + context_emb.shape[1] : -1] = asp_emb[i]
                dataset[i, -1] = 0
                test.append((ent[k]['o_id'], asp[k]['candidate_aspects'][p]['aspect_id'], 0))
                #print(i, k, 'second')
                i += 1
        k += 1

In [19]:
with open(f'{PTH}\\baselinedataset_test.pkl', 'wb') as f:
    pickle.dump(dataset, f)
    print("Dumped")
f.close()

Dumped


In [20]:
import pandas as pd
test_id_df = pd.DataFrame(test, columns = ['Entity ID', 'Aspect ID', 'IsTrue'])

In [98]:
dup = test_id_df.duplicated(subset = ['Entity ID', 'Aspect ID']).tolist()
dup_ind = []
for el in range(len(dup)):
    if dup[el] == True:
        dup_ind.append(el)
        print(el)

In [21]:
test_id_df

,Entity ID,Aspect ID,IsTrue
0,ca4491bc5b58abfa98eaa91481441d49,enwiki:Anatomy/Vertebrate%20anatomy,1
1,ca4491bc5b58abfa98eaa91481441d49,enwiki:Anatomy/Definition,0
2,ca4491bc5b58abfa98eaa91481441d49,enwiki:Anatomy/Animal%20tissues,0
3,ca4491bc5b58abfa98eaa91481441d49,enwiki:Anatomy/Invertebrate%20anatomy,0
4,ca4491bc5b58abfa98eaa91481441d49,enwiki:Anatomy/History,0
...,...,...,...
30585,79f1eebeef5f0203843b11f4ed4b3558,enwiki:Southern%20California%20Rapid%20Transit...,0
30586,79f1eebeef5f0203843b11f4ed4b3558,enwiki:Southern%20California%20Rapid%20Transit...,0
30587,79f1eebeef5f0203843b11f4ed4b3558,enwiki:Southern%20California%20Rapid%20Transit...,0
30588,79f1eebeef5f0203843b11f4ed4b3558,enwiki:Southern%20California%20Rapid%20Transit...,0


In [24]:
test_id_df.to_csv(f'{CSV}\\Test_ID.csv', index = False)